# VoxDoc - Google Colab Deployment

Deploy the VoxDoc Voice Symptom Intake & Documentation Assistant on Google Colab with GPU support.

## Prerequisites
- **Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU)
- **Hugging Face token** with "Read" access ([get one here](https://huggingface.co/settings/tokens))
- Accept model terms for [google/medasr](https://huggingface.co/google/medasr) and [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)
- **ngrok authtoken** ([get one here](https://dashboard.ngrok.com/get-started/your-authtoken))

## What you get
- MedGemma SOAP documentation generation on a free T4 GPU
- MedASR + Whisper speech-to-text
- Medical NER (condition & medication extraction)
- Public URL via ngrok tunnel

## Step 1: Check GPU & Clone Repository

In [ ]:
import torch, subprocess

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB VRAM)")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> Select 'T4 GPU'")
    print("Then re-run this cell.")

# Clone repository (colab-gpu-testing branch)
!rm -rf /content/voice-symptom-documentation-assistant
!git clone --branch colab-gpu-testing https://github.com/JoelJohnsonThomas/voice-symptom-documentation-assistant.git /content/voice-symptom-documentation-assistant
%cd /content/voice-symptom-documentation-assistant
print(f"\nRepo cloned (branch: colab-gpu-testing). Working directory: /content/voice-symptom-documentation-assistant")

## Step 2: Install All Dependencies

This takes ~3-5 minutes. Every package the app needs is installed here — no manual pip installs later.

In [ ]:
import subprocess, sys

def pip(*args):
    """Install packages and show result."""
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  FAILED: pip install {' '.join(args)}")
        if r.stderr:
            print(r.stderr[-1500:])
        return False
    return True

print("1/8  Transformers (MedGemma-compatible commit)...")
pip("git+https://github.com/huggingface/transformers.git@65dc261512cbdb1ee72b88ae5b222f2605aad8e5")

print("2/8  Web framework & networking...")
pip("fastapi==0.110.0", "uvicorn[standard]==0.27.0", "python-multipart==0.0.9",
    "websockets>=12.0", "httpx>=0.27.0")

print("3/8  ML & audio...")
pip("accelerate==0.26.1", "huggingface_hub>=0.24.0")
pip("librosa==0.10.1", "soundfile==0.12.1", "audioread==3.0.1", "noisereduce==3.0.0")
pip("openai-whisper", "--no-deps")  # --no-deps prevents overwriting CUDA torch
pip("more-itertools", "tiktoken", "numba")  # whisper's actual deps, without torch
pip("faster-whisper>=1.0.0")
pip("bitsandbytes")  # optional quantization support

print("4/8  Config & database...")
pip("pydantic==2.6.4", "pydantic-settings==2.2.1", "python-dotenv==1.0.1")
pip("sqlalchemy[asyncio]", "aiosqlite", "alembic>=1.13.0")

print("5/8  Auth & security...")
pip("PyJWT>=2.8.0", "passlib[bcrypt]>=1.7.4", "pyotp>=2.9.0",
    "qrcode[pil]>=7.4", "authlib>=1.3.0")
pip("cryptography>=42.0.0")

print("6/8  Medical NLP (spaCy + SciSpaCy + NER models)...")
pip("spacy>=3.7.0", "scispacy>=0.5.3")
pip("https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz")
pip("https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz")
pip("presidio-analyzer>=2.2.0", "presidio-anonymizer>=2.2.0")

print("7/8  RAG & knowledge base...")
pip("chromadb>=0.5.0", "sentence-transformers>=3.0.0")

print("8/8  Infrastructure & utilities...")
pip("pyngrok>=7.0.0", "nest-asyncio")

# Ensure CUDA torch is still the active version (some deps may have pulled CPU torch)
import torch
if not torch.cuda.is_available():
    print("\n  Reinstalling CUDA torch (a dependency overwrote it)...")
    pip("torch", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu118",
        "--force-reinstall", "--no-deps")

# Verify critical imports
print("\nVerifying critical imports...")
errors = []
for mod, pkg in [
    ("fastapi", "fastapi"), ("uvicorn", "uvicorn"), ("torch", "torch"),
    ("transformers", "transformers"), ("whisper", "openai-whisper"),
    ("librosa", "librosa"), ("sqlalchemy", "sqlalchemy"), ("aiosqlite", "aiosqlite"),
    ("alembic", "alembic"), ("jwt", "PyJWT"), ("passlib", "passlib"),
    ("pyotp", "pyotp"), ("qrcode", "qrcode"), ("authlib", "authlib"),
    ("cryptography", "cryptography"), ("spacy", "spacy"), ("scispacy", "scispacy"),
    ("chromadb", "chromadb"), ("sentence_transformers", "sentence-transformers"),
    ("httpx", "httpx"), ("pyngrok", "pyngrok"),
    ("presidio_analyzer", "presidio-analyzer"),
    ("soundfile", "soundfile"),
]:
    try:
        __import__(mod)
    except ImportError:
        errors.append(f"  MISSING: {mod} (pip install {pkg})")

if errors:
    print("Some packages failed to install:")
    print("\n".join(errors))
else:
    import torch
    cuda_status = f"CUDA {torch.version.cuda}" if torch.cuda.is_available() else "CPU only"
    print(f"All packages installed successfully! (torch: {torch.__version__}, {cuda_status})")

## Step 3: Configure Environment

Replace `HF_TOKEN` and `NGROK_AUTH_TOKEN` with your own values.

In [ ]:
import os
os.chdir("/content/voice-symptom-documentation-assistant")

# ──────────────────────────────────────────────
#  REPLACE THESE TWO VALUES WITH YOUR OWN
# ──────────────────────────────────────────────
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
# ──────────────────────────────────────────────

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

    # Models — whisper-small for high accuracy speech recognition
    f.write("MEDASR_MODEL=google/medasr\n")
    f.write("MEDGEMMA_MODEL=google/medgemma-1.5-4b-it\n")
    f.write("WHISPER_MODEL=openai/whisper-small\n")
    f.write("MODEL_CACHE_DIR=./models\n")

    # GPU — MedGemma runs locally on T4
    f.write("DEVICE=cuda\n")
    f.write("ENABLE_GPU=true\n")
    f.write("COLAB_MODE=true\n")

    # MedGemma provider — local means load on this GPU
    f.write("MEDGEMMA_PROVIDER=local\n")
    f.write("HF_INFERENCE_PROVIDER=\n")
    f.write("HF_INFERENCE_TIMEOUT=120\n")

    # Generation parameters
    f.write("MEDGEMMA_MAX_TOKENS=1024\n")
    f.write("MEDGEMMA_REPETITION_PENALTY=1.1\n")

    # Compliance — REQUIRED or MedGemma won't load
    f.write("MEDGEMMA_TERMS_ACKNOWLEDGED=true\n")
    f.write("ENFORCE_MEDGEMMA_TERMS_ACKNOWLEDGEMENT=true\n")
    f.write("ALLOW_PHI_LOGGING=false\n")
    f.write("ENABLE_PHI_PERSISTENCE=false\n")
    f.write("AUDIT_LOGGING_ENABLED=true\n")

    # Audio — 2s streaming interval (fast on GPU)
    f.write("AUDIO_SAMPLE_RATE=16000\n")
    f.write("MAX_AUDIO_DURATION_SECONDS=300\n")
    f.write("STREAMING_INTERVAL_SECONDS=2.0\n")

    # Rate limiting
    f.write("RATE_LIMITING_ENABLED=true\n")
    f.write("RATE_LIMIT_GENERAL_RPM=60\n")
    f.write("RATE_LIMIT_INFERENCE_RPM=10\n")
    f.write("QUEUE_MAX_CONCURRENT_INFERENCES=2\n")
    f.write("QUEUE_MAX_SIZE=20\n")
    f.write("QUEUE_TIMEOUT_SECONDS=120\n")
    f.write("QUEUE_ESTIMATED_INFERENCE_SECONDS=10\n")

    # Monitoring
    f.write("METRICS_ENABLED=true\n")
    f.write("STRUCTURED_LOGGING_ENABLED=true\n")
    f.write("METRICS_ENDPOINT_AUTH_REQUIRED=false\n")

    # Security (dev defaults — not for production)
    f.write("ENCRYPTION_AT_REST_ENABLED=false\n")
    f.write("ENCRYPTION_MASTER_KEY=CHANGE_ME_IN_PRODUCTION\n")
    f.write("JWT_SECRET_KEY=CHANGE_ME_IN_PRODUCTION\n")

    # RAG (disabled for faster startup)
    f.write("RAG_ENABLED=false\n")
    f.write("RAG_EMBEDDING_MODEL=NeuML/pubmedbert-base-embeddings\n")

    # Logging
    f.write("LOG_LEVEL=INFO\n")

# Configure ngrok
from pyngrok import ngrok
import nest_asyncio
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
nest_asyncio.apply()

print("Environment configured!")
print(f"  Whisper model: openai/whisper-small (high accuracy)")
print(f"  MedGemma: local GPU inference (google/medgemma-1.5-4b-it)")
print(f"  MEDGEMMA_TERMS_ACKNOWLEDGED: true")
print(f"  Streaming interval: 2.0s (fast on GPU)")
print(f"  ngrok: ready")

## Step 4: Start the Server

This will:
1. Load MedASR model on GPU (~30 seconds)
2. Load MedGemma model on GPU (~2-3 minutes, first time downloads ~8 GB)
3. Start the FastAPI server on port 8000
4. Create a public ngrok URL

The ngrok URL will be printed at the top — open it in your browser.

In [ ]:
import os
os.chdir("/content/voice-symptom-documentation-assistant")
os.environ["MEDGEMMA_TERMS_ACKNOWLEDGED"] = "true"

from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"  PUBLIC URL: {public_url}")
print(f"{'='*60}")
print("  Open this URL in your browser to access VoxDoc!")
print(f"{'='*60}\n")

# Start the server (this blocks until you stop the cell)
!cd /content/voice-symptom-documentation-assistant && python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

## Usage & Troubleshooting

### How to use
1. Copy the ngrok URL printed above and open it in your browser
2. Click **Start Recording** or upload an audio file
3. The app transcribes speech and generates SOAP documentation using MedGemma

### To stop
- Click the **Stop** button on the cell above, or press **Ctrl+C**

### Common issues

| Problem | Fix |
|---------|-----|
| `No GPU detected` | Runtime → Change runtime type → T4 GPU, then restart |
| `ModuleNotFoundError` | Re-run the Step 2 install cell |
| `Skipping MedGemma preload: terms not acknowledged` | Make sure `.env` has `MEDGEMMA_TERMS_ACKNOWLEDGED=true` |
| `torch.cuda.is_available() is False` | CUDA torch got overwritten — re-run Step 2 |
| ngrok `ERR_NGROK_108` | Free tier allows 1 tunnel — stop other Colab sessions |
| `en_ner_bc5cdr_md not found` | Re-run Step 2 (installs SciSpaCy models) |
| Documentation shows "N/A" | Model may need warmup — try generating again |

### Notes
- Free Colab sessions last up to 12 hours
- The ngrok URL changes each restart
- Models are cached after first download (faster restarts)
- GPU inference is ~10x faster than CPU